# Build cost table

Produces `data/cost_model/cost_table.csv`. One row per (symbol, venue, week_start) with `half_spread_bps`, `impact_bps_per_unit`, `taker_fee_bps`, and a `half_spread_source` column tagging each row as `book` (measured from bookTicker) or `ar` (Abdi-Ranaldo estimate from local klines).

## Output

One row per (symbol, venue, week_start). Columns.

- `symbol`. Ticker string.
- `venue`. 1 for futures (matches the C++ Portfolio layout).
- `week_start`. ISO Monday of the week, `YYYY-MM-DD`.
- `half_spread_bps`. Half-spread in bps, from bookTicker where available, AR estimate otherwise.
- `half_spread_source`. `"book"` or `"ar"`.
- `impact_bps_per_unit`. Weekly median Amihud illiquidity from aggTrades, bps per unit qty.
- `taker_fee_bps`. 4.0 constant, VIP0 USD-M perps.
- `n_days_book`, `n_days_trade`, `n_bars`. Sample counts contributing to each column.

In [3]:
import io
import time
import zipfile
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/qp-cost-table")
    KLINES_ROOT = None
except ImportError:
    # Walk up until a directory containing "data/binance_historical" is found.
    ROOT = Path.cwd()
    while ROOT != ROOT.parent and not (ROOT / "data" / "binance_historical").is_dir():
        ROOT = ROOT.parent
    KLINES_ROOT = ROOT / "data" / "binance_historical"
    ROOT = ROOT / "data" / "cost_model"

Symbols must match `include/data_source/source/venue/binance/binance_historical/bin_hist_symbol_table.hpp`'s `detail::kSymbols`. 

In [5]:
SYMBOLS   = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
             "DOGEUSDT", "ADAUSDT", "LINKUSDT", "AVAXUSDT", "LTCUSDT"]
FIRST_DAY = date(2022, 1, 1)
LAST_DAY  = date(2024, 12, 31)

# Binance USD-M VIP0 taker fee.
TAKER_FEE_BPS = 4.0
FUTURES_VENUE = 1

BOOK_URL  = "https://data.binance.vision/data/futures/um/daily/bookTicker/{s}/{s}-bookTicker-{d}.zip"
TRADE_URL = "https://data.binance.vision/data/futures/um/daily/aggTrades/{s}/{s}-aggTrades-{d}.zip"

ROOT.mkdir(parents=True, exist_ok=True)

BOOK_CHECKPOINT   = ROOT / "book_daily.csv"
TRADE_CHECKPOINT  = ROOT / "trade_daily.csv"
INTERMEDIATE_PATH = ROOT / "book_trade_weekly.csv"
FINAL_PATH        = ROOT / "cost_table.csv"

Streams each daily zip in memory, extracts, summarizes, discards. Raw never touches disk.

In [ ]:
REQUEST_TIMEOUT = 60
RETRY_SLEEP     = 5

# aggTrades dumps have no header row for most months; some newer months do.
AGG_TRADES_COLS = ["agg_trade_id", "price", "quantity", "first_trade_id",
                   "last_trade_id", "transact_time", "is_buyer_maker"]

def _download_zip(url):
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return r.content
        except requests.RequestException as e:
            if attempt == 2:
                raise
            print(f"  retry {attempt + 1} on {url}: {e}")
            time.sleep(RETRY_SLEEP)

def _read_zipped_csv(zip_bytes, **read_csv_kwargs):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        name = next((n for n in z.namelist() if n.endswith(".csv")), None)
        if name is None:
            return None
        with z.open(name) as f:
            return pd.read_csv(f, **read_csv_kwargs)

def summarize_book_ticker(symbol, day):
    blob = _download_zip(BOOK_URL.format(s=symbol, d=day.isoformat()))
    if blob is None:
        return None
    # Median over ~2k evenly-spaced rows is indistinguishable from
    # the median over ~10M
    df = _read_zipped_csv(blob, skiprows=lambda i: i > 0 and i % 500 != 0)
    if df is None or df.empty:
        return None
    bid = df["best_bid_price"].astype(float)
    ask = df["best_ask_price"].astype(float)
    ok  = (bid > 0) & (ask > 0) & (ask >= bid)
    half_spread_bps = 1e4 * ((ask[ok] - bid[ok]) / (ask[ok] + bid[ok]))
    return {
        "symbol":          symbol,
        "day":             day.isoformat(),
        "half_spread_bps": float(half_spread_bps.median()),
        "n_events":        int(ok.sum()),
    }

def summarize_agg_trades(symbol, day):
    blob = _download_zip(TRADE_URL.format(s=symbol, d=day.isoformat()))
    if blob is None:
        return None
    df = _read_zipped_csv(blob, header=None, names=AGG_TRADES_COLS, low_memory=False)
    if df is None or df.empty:
        return None
    # Drop a header row when a month happens to have one.
    try:
        float(df.iloc[0]["price"])
    except (ValueError, TypeError):
        df = df.iloc[1:]
    if df.empty:
        return None
    price = pd.to_numeric(df["price"], errors="coerce").to_numpy()
    qty   = pd.to_numeric(df["quantity"], errors="coerce").to_numpy()
    ts    = pd.to_numeric(df["transact_time"], errors="coerce").to_numpy()
    order = np.argsort(ts)
    price = price[order]
    qty   = qty[order]
    if len(price) < 2:
        return None
    with np.errstate(divide="ignore", invalid="ignore"):
        log_ret  = np.abs(np.diff(np.log(price)))
        per_unit = log_ret / np.where(qty[:-1] > 0, qty[:-1], np.nan)
    per_unit = per_unit[np.isfinite(per_unit)]
    if per_unit.size == 0:
        return None
    return {
        "symbol":              symbol,
        "day":                 day.isoformat(),
        "impact_bps_per_unit": float(1e4 * np.median(per_unit)),
        "n_trades":            int(len(price)),
    }

Iterates `(symbol, day, kind)`, resumes from checkpoints, flushes per symbol.

In [ ]:
def _load_or_empty(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

def _done_pairs(df):
    if df.empty:
        return set()
    return set(zip(df["symbol"], df["day"]))

def _days_in_range(first, last):
    d = first
    while d <= last:
        yield d
        d = d + timedelta(days=1)

def run(kinds=("book", "trade"), symbols=None):
    syms = symbols if symbols is not None else SYMBOLS

    book_rows  = _load_or_empty(BOOK_CHECKPOINT).to_dict("records")
    trade_rows = _load_or_empty(TRADE_CHECKPOINT).to_dict("records")
    book_done  = _done_pairs(_load_or_empty(BOOK_CHECKPOINT))
    trade_done = _done_pairs(_load_or_empty(TRADE_CHECKPOINT))

    for symbol in syms:
        print(f"=== {symbol} ===")
        for day in _days_in_range(FIRST_DAY, LAST_DAY):
            key = (symbol, day.isoformat())
            if "book" in kinds and key not in book_done:
                row = summarize_book_ticker(symbol, day)
                if row is not None:
                    book_rows.append(row)
            if "trade" in kinds and key not in trade_done:
                row = summarize_agg_trades(symbol, day)
                if row is not None:
                    trade_rows.append(row)
        if "book" in kinds:
            pd.DataFrame(book_rows).to_csv(BOOK_CHECKPOINT, index=False)
        if "trade" in kinds:
            pd.DataFrame(trade_rows).to_csv(TRADE_CHECKPOINT, index=False)

run(kinds=("book", "trade"))

Reads the per-day checkpoints, buckets by ISO Monday, medians per (symbol, week), joins book and trade, writes `book_trade_weekly.csv`.

In [6]:
def _week_start(day_str):
    d = date.fromisoformat(day_str)
    return (d - timedelta(days=d.weekday())).isoformat()

def build_intermediate_table():
    book  = _load_or_empty(BOOK_CHECKPOINT)
    trade = _load_or_empty(TRADE_CHECKPOINT)
    if book.empty and trade.empty:
        raise RuntimeError("no checkpoints yet, run() first")

    if not book.empty:
        book = book.copy()
        book["week_start"] = book["day"].map(_week_start)
        book_weekly = book.groupby(["symbol", "week_start"], as_index=False).agg(
            half_spread_bps_book=("half_spread_bps", "median"),
            n_days_book=("day", "count"),
        )
    else:
        book_weekly = pd.DataFrame(columns=["symbol", "week_start",
                                            "half_spread_bps_book", "n_days_book"])

    if not trade.empty:
        trade = trade.copy()
        trade["week_start"] = trade["day"].map(_week_start)
        trade_weekly = trade.groupby(["symbol", "week_start"], as_index=False).agg(
            impact_bps_per_unit=("impact_bps_per_unit", "median"),
            n_days_trade=("day", "count"),
        )
    else:
        trade_weekly = pd.DataFrame(columns=["symbol", "week_start",
                                             "impact_bps_per_unit", "n_days_trade"])

    weekly = book_weekly.merge(trade_weekly, on=["symbol", "week_start"], how="outer")
    weekly["venue"]         = FUTURES_VENUE
    weekly["taker_fee_bps"] = TAKER_FEE_BPS

    weekly = weekly[["symbol", "venue", "week_start", "half_spread_bps_book",
                     "impact_bps_per_unit", "taker_fee_bps",
                     "n_days_book", "n_days_trade"]]
    weekly = weekly.sort_values(["symbol", "week_start"]).reset_index(drop=True)

    weekly.to_csv(INTERMEDIATE_PATH, index=False)
    print(f"wrote {INTERMEDIATE_PATH}  rows={len(weekly)}")
    return weekly

build_intermediate_table()

NameError: name '_load_or_empty' is not defined

## Abdi-Ranaldo half-spread from klines

Reads local 1 min futures klines, computes Abdi-Ranaldo half spread per (symbol, ISO-week).

$s^2 = 4 E[(c_t - eta_t)(c_t - eta_{t+1})]$ where $eta_t = ln((H_t + L_t) / 2)$ and $c_t = ln(close_t)$.

Half spread is $s / 2 = sqrt(E[...])$.

In [7]:
KLINE_COLS = ["symbol", "kind", "open_time", "open", "high", "low", "close",
              "volume", "close_time", "quote_volume", "count",
              "taker_buy_base", "taker_buy_quote", "ignore"]

def _load_klines(symbol):
    if KLINES_ROOT is None:
        raise RuntimeError("Klines not found")
    d = KLINES_ROOT / symbol / "futures" / "klines"
    files = sorted(d.glob(f"{symbol}-1m-*.csv"))
    if not files:
        raise FileNotFoundError(f"no klines under {d}")
    parts = []
    for f in files:
        parts.append(pd.read_csv(f, header=None, names=KLINE_COLS,
                                 usecols=["open_time", "high", "low", "close"]))
    df = pd.concat(parts, ignore_index=True)
    df["ts"]    = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["high"]  = df["high"].astype(float)
    df["low"]   = df["low"].astype(float)
    df["close"] = df["close"].astype(float)
    df = df.sort_values("ts").drop_duplicates("ts").reset_index(drop=True)
    return df[["ts", "high", "low", "close"]]

def _week_start_from_ts(ts):
    d = ts.date()
    return (d - timedelta(days=d.weekday())).isoformat()

def _ar_by_week(klines):
    k   = klines.loc[(klines["high"] > 0) & (klines["low"] > 0)
                     & (klines["close"] > 0)].reset_index(drop=True)
    eta = np.log((k["high"].to_numpy() + k["low"].to_numpy()) / 2.0)
    c   = np.log(k["close"].to_numpy())
    term = (c[:-1] - eta[:-1]) * (c[:-1] - eta[1:])
    bucket = k["ts"].iloc[:-1].map(_week_start_from_ts).to_numpy()
    df = pd.DataFrame({"week_start": bucket, "term": term})
    weekly = df.groupby("week_start", as_index=False).agg(
        mean_term=("term", "mean"),
        n_bars=("term", "count"),
    )
    weekly["half_spread_bps_ar"] = 1e4 * np.sqrt(np.clip(weekly["mean_term"], 0.0, None))
    return weekly[["week_start", "half_spread_bps_ar", "n_bars"]]

def build_ar_table(symbols=None):
    rows = []
    for symbol in (symbols or SYMBOLS):
        print(f"  ar: {symbol}")
        w = _ar_by_week(_load_klines(symbol))
        w["symbol"] = symbol
        w["venue"]  = FUTURES_VENUE
        rows.append(w)
    return pd.concat(rows, ignore_index=True)

ar_table = build_ar_table()
ar_table.head()

  ar: BTCUSDT
  ar: ETHUSDT
  ar: SOLUSDT
  ar: BNBUSDT
  ar: XRPUSDT
  ar: DOGEUSDT
  ar: ADAUSDT
  ar: LINKUSDT
  ar: AVAXUSDT
  ar: LTCUSDT


,week_start,half_spread_bps_ar,n_bars,symbol,venue
0,2021-12-27,0.206829,2879,BTCUSDT,1
1,2022-01-03,0.000000,10073,BTCUSDT,1
2,2022-01-10,0.000000,10073,BTCUSDT,1
3,2022-01-17,0.000000,10073,BTCUSDT,1
4,2022-01-24,0.000000,10073,BTCUSDT,1


# Calibration and merge

Reads `data/cost_model/book_trade_weekly.csv`, joins with AR, prints per-symbol book-vs-AR agreement over the overlap window.

In [11]:
def _calibrate(merged):
    overlap = merged.dropna(subset=["half_spread_bps_book", "half_spread_bps_ar"])
    if overlap.empty:
        print("no overlap between book and AR windows; skipping calibration")
        return
    print()
    print("=== calibration (book vs AR over overlap weeks) ===")
    print(f"{'symbol':10s} {'weeks':>6s} {'corr':>6s} {'book_med':>10s} {'ar_med':>10s} {'ratio':>7s}")
    for symbol, g in overlap.groupby("symbol"):
        b = g["half_spread_bps_book"].to_numpy()
        a = g["half_spread_bps_ar"].to_numpy()
        corr  = float(np.corrcoef(b, a)[0, 1]) if len(g) > 2 else float("nan")
        b_med = float(np.median(b))
        a_med = float(np.median(a))
        ratio = a_med / b_med if b_med > 0 else float("nan")
        print(f"{symbol:10s} {len(g):>6d} {corr:>6.2f} {b_med:>10.3f} {a_med:>10.3f} {ratio:>7.2f}")
    print("=====================================================")


def finalize():
    if not INTERMEDIATE_PATH.exists():
        raise FileNotFoundError(
            f"expected {INTERMEDIATE_PATH}. Move the Colab output there first.")

    inter = pd.read_csv(INTERMEDIATE_PATH)
    ar    = build_ar_table()

    merged = inter.merge(ar, on=["symbol", "venue", "week_start"], how="outer")

    # Report AR for the record
    _calibrate(merged)

    book_median = merged.groupby("symbol")["half_spread_bps_book"].transform("median")

    b = merged["half_spread_bps_book"]
    merged["half_spread_bps"]    = b.where(b.notna(), book_median)
    merged["half_spread_source"] = np.where(b.notna(), "book", "book_median")
    merged["taker_fee_bps"]      = merged["taker_fee_bps"].fillna(TAKER_FEE_BPS)
    merged["venue"]              = merged["venue"].fillna(FUTURES_VENUE).astype(int)

    final = merged[["symbol", "venue", "week_start", "half_spread_bps",
                    "half_spread_source", "impact_bps_per_unit",
                    "taker_fee_bps", "n_days_book", "n_days_trade", "n_bars"]]
    final = final.sort_values(["symbol", "week_start"]).reset_index(drop=True)

    final.to_csv(FINAL_PATH, index=False)
    print(f"\nwrote {FINAL_PATH}  rows={len(final)}")
    print(f"provenance: {dict(final['half_spread_source'].value_counts())}")
    print(f"nan half_spread: {int(final['half_spread_bps'].isna().sum())}")
    return final


finalize()

  ar: BTCUSDT
  ar: ETHUSDT
  ar: SOLUSDT
  ar: BNBUSDT
  ar: XRPUSDT
  ar: DOGEUSDT
  ar: ADAUSDT
  ar: LINKUSDT
  ar: AVAXUSDT
  ar: LTCUSDT

=== calibration (book vs AR over overlap weeks) ===
symbol      weeks   corr   book_med     ar_med   ratio
ADAUSDT        46   0.63      1.369      1.349    0.99
AVAXUSDT       46   0.01      0.359      0.000    0.00
BNBUSDT        46  -0.18      0.205      0.000    0.00
BTCUSDT        46  -0.47      0.016      0.000    0.00
DOGEUSDT       46   0.01      0.667      0.380    0.57
ETHUSDT        46  -0.30      0.026      0.000    0.00
LINKUSDT       46  -0.00      0.541      0.292    0.54
LTCUSDT        46   0.21      0.694      0.352    0.51
SOLUSDT        46  -0.31      0.198      0.000    0.00
XRPUSDT        46   0.13      0.897      0.408    0.45

wrote /home/danie/quant-platform/data/cost_model/cost_table.csv  rows=1580
provenance: {'book_median': np.int64(1120), 'book': np.int64(460)}
nan half_spread: 0


,symbol,venue,week_start,half_spread_bps,half_spread_source,impact_bps_per_unit,taker_fee_bps,n_days_book,n_days_trade,n_bars
0,ADAUSDT,1,2021-12-27,1.368747,book_median,0.001085,4.0,NaN,2,2880
1,ADAUSDT,1,2022-01-03,1.368747,book_median,0.001677,4.0,NaN,7,10073
2,ADAUSDT,1,2022-01-10,1.368747,book_median,0.001397,4.0,NaN,7,10073
3,ADAUSDT,1,2022-01-17,1.368747,book_median,0.001278,4.0,NaN,7,10073
4,ADAUSDT,1,2022-01-24,1.368747,book_median,0.001113,4.0,NaN,7,10073
...,...,...,...,...,...,...,...,...,...,...
1575,XRPUSDT,1,2024-12-02,0.896505,book_median,0.000664,4.0,NaN,7,10080
1576,XRPUSDT,1,2024-12-09,0.896505,book_median,0.000516,4.0,NaN,7,10080
1577,XRPUSDT,1,2024-12-16,0.896505,book_median,0.000586,4.0,NaN,7,10080
1578,XRPUSDT,1,2024-12-23,0.896505,book_median,0.000418,4.0,NaN,7,10080


## AR outcome

Ran Abdi-Ranaldo over the 46-week bookTicker overlap window on all 10 symbols. AR is not usable on this data.

- BTC, ETH, SOL, BNB, AVAX. The covariance term goes negative and the estimator clips to zero across most weeks.
- LINK, LTC, XRP, DOGE. Estimator produces roughly half the measured spread with no meaningful correlation to the book values (corr ~0).
- ADA. Only symbol where AR tracks book (corr 0.63, ratio 0.99). One out of ten is not enough to trust.

## Fallback: per-symbol book median

Filling the 22 months outside the bookTicker archive with a per-symbol constant equal to the median of that symbol's 46 measured book weeks. Loses time variation in the gap window but is grounded in real measurements and does not produce zero-cost fills. Provenance column marks each row as either `book` (measured for that week) or `book_median` (per-symbol constant fill). The AR helper stays in the notebook in case a future dataset (longer bookTicker window, wider-spread symbol universe) makes it viable.
